# Apex Retail Intelligence — Phase 5 
## Gold Layer (Star Schema) 

---

### Objectives

| Phase | Deliverable | Detail |
| --- | --- | --- |
| 5 | Gold Layer (Star Schema) | Dimension & fact tables optimized for analytics |


---

### Star Schema Design

```
          ┌──────────────┐
          │  dim_customer │
          └──────┬───────┘
                 │
┌────────────┐   │   ┌─────────────┐
│ dim_product ├───┼───┤  fact_sales  │
└────────────┘   │   └─────────────┘
                 │
          ┌──────┴───────┐
          │   dim_date    │
          └──────────────┘
```

---

### KPI Metrics

* **Revenue** — Total sales amount by category, region, and time period
* **Customer LTV** — Lifetime spend, order frequency, avg order value
* **Product Performance** — Units sold, revenue contribution, sell-through rate
* **Regional Analysis** — Sales distribution by city/state

2 – Widgets & Schema

In [0]:
dbutils.widgets.text("silver_catalog", "apex_retail", "Silver Catalog")
dbutils.widgets.text("silver_schema", "silver", "Silver Schema")
dbutils.widgets.text("gold_catalog", "apex_retail", "Gold Catalog")
dbutils.widgets.text("gold_schema", "GOLD_tables", "Gold Schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}")

from pyspark.sql import functions as F

Create dim_customer

In [0]:
# dim_customer — SQL DDL from Silver (already has SCD2 + surrogate key)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.dim_customer AS
SELECT
  customer_sk, customer_id, first_name, last_name,
  email, phone, city, state, signup_date,
  effective_start_date, effective_end_date, is_active
FROM delta.`/Volumes/apex_retail/landing/silver/customer`
""")

dim_customer = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_customer")
print(f"dim_customer: {dim_customer.count()} rows | Active: {dim_customer.filter(F.col('is_active')==True).count()}")
display(dim_customer.limit(5))

dim_customer: 60 rows | Active: 60


customer_sk,customer_id,first_name,last_name,email,phone,city,state,signup_date,effective_start_date,effective_end_date,is_active
1,C0001,Alice,Smith,cust1@example.com,555-1000,New York,NY,2023-01-01,2023-01-01,9999-12-31,true
2,C0002,Bob,Jones,cust2@example.com,555-1001,Chicago,IL,2023-01-08,2023-01-08,9999-12-31,true
3,C0003,Carol,Brown,cust3@example.com,555-1002,Houston,TX,2023-01-15,2023-01-15,9999-12-31,true
4,C0004,Dave,Wilson,cust4@example.com,555-1003,Phoenix,AZ,2023-01-22,2023-01-22,9999-12-31,true
5,C0005,Eve,Taylor,cust5@example.com,555-1004,Dallas,TX,2023-01-29,2023-01-29,9999-12-31,true


Create dim_product

In [0]:
# dim_product — SQL DDL from Silver (already has product_sk)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.dim_product AS
SELECT
  product_sk, product_id, product_name,
  category, price, stock_quantity
FROM delta.`/Volumes/apex_retail/landing/silver/product`
""")

dim_product = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_product")
print(f"dim_product: {dim_product.count()} rows")
display(dim_product.limit(5))

dim_product: 35 rows


product_sk,product_id,product_name,category,price,stock_quantity
1,P0001,Product_B1,Electronics,15.50,110
2,P0002,Product_C2,Clothing,21.00,120
3,P0003,Product_D3,Home,26.50,130
4,P0004,Product_E4,Sports,32.00,140
5,P0005,Product_F5,Food,37.50,150


Create dim_promotion

In [0]:
# dim_promotion — SQL DDL placeholder (no promotion data in source)
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.dim_promotion AS
SELECT
  1 AS promotion_sk,
  'NONE' AS promotion_id,
  'No Promotion' AS promotion_type
""")

dim_promotion = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_promotion")
print(f"dim_promotion: {dim_promotion.count()} rows (placeholder — no promotion data in source)")
display(dim_promotion)

dim_promotion: 1 rows (placeholder — no promotion data in source)


promotion_sk,promotion_id,promotion_type
1,NONE,No Promotion


Create dim_date

In [0]:
# dim_date — build from ALL Silver sales dates using SQL DDL
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.dim_date AS
SELECT DISTINCT
  CAST(date_format(sale_date, 'yyyyMMdd') AS INT) AS date_sk,
  sale_date AS full_date,
  YEAR(sale_date) AS year,
  MONTH(sale_date) AS month,
  DAY(sale_date) AS day,
  date_format(sale_date, 'EEEE') AS day_of_week,
  weekofyear(sale_date) AS week_of_year,
  QUARTER(sale_date) AS quarter,
  CASE WHEN dayofweek(sale_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM delta.`/Volumes/apex_retail/landing/silver/sales`
""")

dim_date = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_date")
print(f"dim_date: {dim_date.count()} rows")
display(dim_date)

dim_date: 120 rows


date_sk,full_date,year,month,day,day_of_week,week_of_year,quarter,is_weekend
20230316,2023-03-16,2023,3,16,Thursday,11,1,false
20230328,2023-03-28,2023,3,28,Tuesday,13,1,false
20230503,2023-05-03,2023,5,3,Wednesday,18,2,false
20230509,2023-05-09,2023,5,9,Tuesday,19,2,false
20230623,2023-06-23,2023,6,23,Friday,25,2,false
20230711,2023-07-11,2023,7,11,Tuesday,28,3,false
20230804,2023-08-04,2023,8,4,Friday,31,3,false
20230810,2023-08-10,2023,8,10,Thursday,32,3,false
20230825,2023-08-25,2023,8,25,Friday,34,3,false
20231006,2023-10-06,2023,10,6,Friday,40,4,false


Cell 7 – Load Dimensions

In [0]:
# Reload Gold tables for fact_sales build
sales = spark.read.format("delta").load("/Volumes/apex_retail/landing/silver/sales")
cust = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_customer")
prod = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_product")
promo = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_promotion")
dates = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_date")

print(f"Silver sales: {sales.count()} | dim_customer: {cust.count()} | dim_product: {prod.count()} | dim_date: {dates.count()}")

Silver sales: 120 | dim_customer: 60 | dim_product: 35 | dim_date: 120


Create fact_sales

In [0]:
# fact_sales — SQL DDL with surrogate key joins to all dimensions
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.fact_sales AS
SELECT
  s.sales_sk,
  s.sale_id,
  c.customer_sk,
  p.product_sk,
  1 AS promotion_sk,
  d.date_sk,
  s.sale_date,
  s.quantity,
  s.total_amount
FROM delta.`/Volumes/apex_retail/landing/silver/sales` s
LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.dim_customer c
  ON s.customer_id = c.customer_id AND c.is_active = TRUE
LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.dim_product p
  ON s.product_id = p.product_id
LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.dim_date d
  ON s.sale_date = d.full_date
""")

fact_sales = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.fact_sales")
print(f"fact_sales: {fact_sales.count()} rows")
print(f"  customer_sk populated: {fact_sales.filter(F.col('customer_sk').isNotNull()).count()}")
print(f"  product_sk populated:  {fact_sales.filter(F.col('product_sk').isNotNull()).count()}")
print(f"  date_sk populated:     {fact_sales.filter(F.col('date_sk').isNotNull()).count()}")
display(fact_sales.limit(5))

fact_sales: 120 rows
  customer_sk populated: 120
  product_sk populated:  120
  date_sk populated:     120


sales_sk,sale_id,customer_sk,product_sk,promotion_sk,date_sk,sale_date,quantity,total_amount
1,S00001,41,15,1,20230115,2023-01-15,2,99.91
2,S00002,8,13,1,20230118,2023-01-18,5,236.69
3,S00003,2,9,1,20230121,2023-01-21,1,438.50
4,S00004,48,30,1,20230124,2023-01-24,3,46.94
5,S00005,18,21,1,20230127,2023-01-27,5,405.93


Cell 9 – Verify Gold Tables

In [0]:
spark.sql(f"SHOW TABLES IN {GOLD_CATALOG}.{GOLD_SCHEMA}")

display(spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_customer"))
display(spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_product"))
display(spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_promotion"))
display(spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_date"))
display(spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.fact_sales"))

customer_sk,customer_id,first_name,last_name,email,phone,city,state,signup_date,effective_start_date,effective_end_date,is_active
1,C0001,Alice,Smith,cust1@example.com,555-1000,New York,NY,2023-01-01,2023-01-01,9999-12-31,true
2,C0002,Bob,Jones,cust2@example.com,555-1001,Chicago,IL,2023-01-08,2023-01-08,9999-12-31,true
3,C0003,Carol,Brown,cust3@example.com,555-1002,Houston,TX,2023-01-15,2023-01-15,9999-12-31,true
4,C0004,Dave,Wilson,cust4@example.com,555-1003,Phoenix,AZ,2023-01-22,2023-01-22,9999-12-31,true
5,C0005,Eve,Taylor,cust5@example.com,555-1004,Dallas,TX,2023-01-29,2023-01-29,9999-12-31,true
6,C0006,Frank,Clark,cust6@example.com,555-1005,New York,NY,2023-02-05,2023-02-05,9999-12-31,true
7,C0007,Grace,Hall,cust7@example.com,555-1006,Chicago,IL,2023-02-12,2023-02-12,9999-12-31,true
8,C0008,Hank,Adams,cust8@example.com,555-1007,Houston,TX,2023-02-19,2023-02-19,9999-12-31,true
9,C0009,Iris,White,cust9@example.com,555-1008,Phoenix,AZ,2023-02-26,2023-02-26,9999-12-31,true
10,C0010,Jack,Green,cust10@example.com,555-1009,Dallas,TX,2023-03-05,2023-03-05,9999-12-31,true


product_sk,product_id,product_name,category,price,stock_quantity
1,P0001,Product_B1,Electronics,15.50,110
2,P0002,Product_C2,Clothing,21.00,120
3,P0003,Product_D3,Home,26.50,130
4,P0004,Product_E4,Sports,32.00,140
5,P0005,Product_F5,Food,37.50,150
6,P0006,Product_G6,Electronics,43.00,160
7,P0007,Product_H7,Clothing,48.50,170
8,P0008,Product_I8,Home,54.00,180
9,P0009,Product_J9,Sports,59.50,190
10,P0010,Product_K10,Food,65.00,200


promotion_sk,promotion_id,promotion_type
1,NONE,No Promotion


date_sk,full_date,year,month,day,day_of_week,week_of_year,quarter,is_weekend
20230316,2023-03-16,2023,3,16,Thursday,11,1,false
20230328,2023-03-28,2023,3,28,Tuesday,13,1,false
20230503,2023-05-03,2023,5,3,Wednesday,18,2,false
20230509,2023-05-09,2023,5,9,Tuesday,19,2,false
20230623,2023-06-23,2023,6,23,Friday,25,2,false
20230711,2023-07-11,2023,7,11,Tuesday,28,3,false
20230804,2023-08-04,2023,8,4,Friday,31,3,false
20230810,2023-08-10,2023,8,10,Thursday,32,3,false
20230825,2023-08-25,2023,8,25,Friday,34,3,false
20231006,2023-10-06,2023,10,6,Friday,40,4,false


sales_sk,sale_id,customer_sk,product_sk,promotion_sk,date_sk,sale_date,quantity,total_amount
1,S00001,41,15,1,20230115,2023-01-15,2,99.91
2,S00002,8,13,1,20230118,2023-01-18,5,236.69
3,S00003,2,9,1,20230121,2023-01-21,1,438.50
4,S00004,48,30,1,20230124,2023-01-24,3,46.94
5,S00005,18,21,1,20230127,2023-01-27,5,405.93
6,S00006,16,23,1,20230130,2023-01-30,5,429.42
7,S00007,15,18,1,20230202,2023-02-02,2,57.98
8,S00008,9,8,1,20230205,2023-02-05,2,329.55
9,S00009,48,22,1,20230208,2023-02-08,3,274.89
10,S00010,7,11,1,20230211,2023-02-11,2,17.23


---

## Phase 6 — Business Reporting (KPI Generation)

**Important Constraint:** No external dashboards (e.g., Power BI, Tableau). Deliverable = PySpark logic + outputs rendered within Databricks notebook.

| # | KPI Name | Definition |
| --- | --- | --- |
| 1 | Net Margin by Region | Total gross revenue minus discounts, grouped by store region |
| 2 | AOV by Promotion | Which promotion types drive the highest average cart values |
| 3 | Demographic Churn Heatmap | Customer churn rates split by state and loyalty programme |
| 4 | Product Quality Index | Which product categories suffer the highest return rates |
| 5 | Store Traffic by Hour | Busiest transaction hours and days of the week |

In [0]:
# ==============================================================
# KPI 1: Net Margin by Region
# Definition: Total gross revenue minus discounts, grouped by store region.
# Note: Source data does not include a 'discount' column.
#       net_margin = revenue - discount (discount assumed 0)
# ==============================================================

fact = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.fact_sales")
cust = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_customer")

net_margin_by_region = (
    fact.alias("f")
    .join(
        cust.filter(F.col("is_active") == True).alias("c"),
        F.col("f.customer_sk") == F.col("c.customer_sk"),
        "left"
    )
    .groupBy("c.state", "c.city")
    .agg(
        F.round(F.sum("f.total_amount"), 2).alias("gross_revenue"),
        F.lit(0.00).alias("total_discount"),
        F.round(F.sum("f.total_amount") - F.lit(0), 2).alias("net_margin"),
        F.count("f.sale_id").alias("total_orders"),
    )
    .withColumn("margin_pct", F.round(F.col("net_margin") / F.col("gross_revenue") * 100, 2))
    .orderBy(F.desc("net_margin"))
)

print("KPI 1: Net Margin by Region")
print("Note: Discount assumed 0 — source data lacks discount field")
display(net_margin_by_region)

KPI 1: Net Margin by Region
Note: Discount assumed 0 — source data lacks discount field


state,city,gross_revenue,total_discount,net_margin,total_orders,margin_pct
TX,Houston,7771.59,0.0,7771.59,28,100.00
NY,New York,6960.26,0.0,6960.26,26,100.00
TX,Dallas,5328.05,0.0,5328.05,27,100.00
IL,Chicago,4458.34,0.0,4458.34,18,100.00
AZ,Phoenix,3868.79,0.0,3868.79,19,100.00
MA,Boston,395.35,0.0,395.35,1,100.00
CO,Denver,302.32,0.0,302.32,1,100.00


In [0]:
# ==============================================================
# KPI 2: Average Order Value (AOV) by Promotion
# Definition: Identify which promotion types drive the highest
#             average cart values.
# ==============================================================

promo = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_promotion")

aov_by_promotion = (
    fact.alias("f")
    .join(
        promo.alias("pr"),
        F.col("f.promotion_sk") == F.col("pr.promotion_sk"),
        "left"
    )
    .groupBy("pr.promotion_type", "pr.promotion_id")
    .agg(
        F.count("f.sale_id").alias("total_orders"),
        F.sum("f.quantity").alias("total_units"),
        F.round(F.avg("f.total_amount"), 2).alias("avg_order_value"),
        F.round(F.sum("f.total_amount"), 2).alias("total_revenue"),
    )
    .orderBy(F.desc("avg_order_value"))
)

print("KPI 2: Average Order Value (AOV) by Promotion")
display(aov_by_promotion)

KPI 2: Average Order Value (AOV) by Promotion


promotion_type,promotion_id,total_orders,total_units,avg_order_value,total_revenue
No Promotion,NONE,120,337,242.37,29084.70


In [0]:
# ==============================================================
# KPI 3: Demographic Churn Heatmap
# Definition: Analyse customer churn rates split by state and
#             loyalty programme membership.
# Approximation: Churn = no purchase in last 30 days from max date.
#               Loyalty tier = signup tenure proxy.
# ==============================================================

from pyspark.sql.functions import datediff, max as _max, when

max_date = fact.agg(_max("sale_date")).collect()[0][0]

customer_recency = (
    fact.alias("f")
    .join(
        cust.filter(F.col("is_active") == True).alias("c"),
        F.col("f.customer_sk") == F.col("c.customer_sk"),
        "inner"
    )
    .groupBy("c.customer_sk", "c.state", "c.signup_date")
    .agg(_max("f.sale_date").alias("last_purchase"))
    .withColumn("days_since_purchase", datediff(F.lit(max_date), F.col("last_purchase")))
    .withColumn("is_churned", when(F.col("days_since_purchase") > 30, True).otherwise(False))
    .withColumn(
        "loyalty_tier",
        when(datediff(F.lit(max_date), F.col("signup_date")) > 60, "Early Adopter")
        .when(datediff(F.lit(max_date), F.col("signup_date")) > 30, "Regular")
        .otherwise("New")
    )
)

churn_heatmap = (
    customer_recency
    .groupBy("state", "loyalty_tier")
    .agg(
        F.count("*").alias("total_customers"),
        F.sum(F.when(F.col("is_churned"), 1).otherwise(0)).alias("churned_customers"),
    )
    .withColumn("churn_rate_pct", F.round(F.col("churned_customers") / F.col("total_customers") * 100, 2))
    .orderBy(F.desc("churn_rate_pct"))
)

print(f"KPI 3: Demographic Churn Heatmap (State x Loyalty Tier)")
print(f"Reference date: {max_date} | Churn threshold: 30 days")
display(churn_heatmap)

KPI 3: Demographic Churn Heatmap (State x Loyalty Tier)
Reference date: 2025-02-08 | Churn threshold: 30 days


state,loyalty_tier,total_customers,churned_customers,churn_rate_pct
AZ,Early Adopter,8,7,87.5
IL,Early Adopter,8,7,87.5
TX,Early Adopter,19,12,63.16
NY,Early Adopter,7,3,42.86
CO,Regular,1,0,0.0
MA,Regular,1,0,0.0


In [0]:
# ==============================================================
# KPI 4: Product Quality Index
# Definition: Determine which product categories suffer the
#             highest return rates.
# Approximation: Source data lacks returns. Proxy = inverse
#   quality score using low AOV + high stddev as quality concern.
# ==============================================================

prod = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_product")

product_quality = (
    fact.alias("f")
    .join(
        prod.alias("p"),
        F.col("f.product_sk") == F.col("p.product_sk"),
        "left"
    )
    .groupBy("p.category", "p.product_name", "p.product_id")
    .agg(
        F.count("f.sale_id").alias("total_orders"),
        F.sum("f.quantity").alias("total_units_sold"),
        F.round(F.avg("f.total_amount"), 2).alias("avg_order_value"),
        F.round(F.stddev("f.total_amount"), 2).alias("order_value_stddev"),
        F.lit(0.0).alias("return_rate"),
    )
    .withColumn(
        "quality_index",
        F.round(
            (F.col("avg_order_value") / (F.col("order_value_stddev") + F.lit(1))) * 10, 2
        )
    )
    .orderBy("quality_index")
)

print("KPI 4: Product Quality Index")
print("Note: Return rate = 0 (no returns data). Quality index = AOV / (StdDev + 1) * 10")
print("Lower index = higher price variance (potential quality concern)")
display(product_quality)

KPI 4: Product Quality Index
Note: Return rate = 0 (no returns data). Quality index = AOV / (StdDev + 1) * 10
Lower index = higher price variance (potential quality concern)


category,product_name,product_id,total_orders,total_units_sold,avg_order_value,order_value_stddev,return_rate,quality_index
Electronics,Product_G6,P0006,2,4,180.72,234.98,0.0,7.66
Electronics,Product_L11,P0011,5,9,151.66,166.5,0.0,9.05
Home,Product_S18,P0018,5,18,96.36,100.84,0.0,9.46
Clothing,Product_B27,P0027,4,14,252.60,244.35,0.0,10.3
Food,Product_K10,P0010,3,7,153.18,133.71,0.0,11.37
Food,Product_E30,P0030,3,10,187.31,153.57,0.0,12.12
Sports,Product_T19,P0019,3,8,289.60,231.14,0.0,12.48
Electronics,Product_A26,P0026,2,2,202.52,158.18,0.0,12.72
Clothing,Product_C2,P0002,3,8,245.16,191.59,0.0,12.73
Food,Product_F5,P0005,4,9,282.15,215.03,0.0,13.06


In [0]:
# ==============================================================
# KPI 5: Store Traffic by Hour
# Definition: Identify the busiest transaction hours and days
#             of the week for store foot traffic.
# Approximation: Source data lacks transaction_hour.
#               Showing traffic patterns by day of week.
# ==============================================================

dim_d = spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.dim_date")

traffic_by_day = (
    fact.alias("f")
    .join(
        dim_d.alias("d"),
        F.col("f.date_sk") == F.col("d.date_sk"),
        "inner"
    )
    .groupBy("d.day_of_week", "d.is_weekend")
    .agg(
        F.count("f.sale_id").alias("transaction_count"),
        F.sum("f.quantity").alias("total_units"),
        F.round(F.sum("f.total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("f.total_amount"), 2).alias("avg_transaction_value"),
    )
    .withColumn(
        "traffic_level",
        F.when(F.col("transaction_count") >= 3, "High")
        .when(F.col("transaction_count") >= 2, "Medium")
        .otherwise("Low")
    )
    .orderBy(F.desc("transaction_count"))
)

print("KPI 5: Store Traffic by Day of Week")
print("Note: transaction_hour not in source; showing day-level traffic patterns.")
display(traffic_by_day)

KPI 5: Store Traffic by Day of Week
Note: transaction_hour not in source; showing day-level traffic patterns.


day_of_week,is_weekend,transaction_count,total_units,total_revenue,avg_transaction_value,traffic_level
Sunday,true,18,51,3563.67,197.98,High
Wednesday,false,18,53,4758.36,264.35,High
Tuesday,false,17,47,4063.24,239.01,High
Saturday,true,17,41,4612.97,271.35,High
Thursday,false,17,43,3421.79,201.28,High
Friday,false,17,54,4605.72,270.92,High
Monday,false,16,48,4058.95,253.68,High


In [0]:
print("\n" + "="*60)
print("   PHASE 6: BUSINESS REPORTING (KPI GENERATION) COMPLETE")
print("="*60)
print("")
print("  KPI 1: Net Margin by Region              ✓")
print("  KPI 2: AOV by Promotion                  ✓")
print("  KPI 3: Demographic Churn Heatmap         ✓")
print("  KPI 4: Product Quality Index             ✓")
print("  KPI 5: Store Traffic by Day of Week      ✓")
print("")
print("  All KPIs computed using PySpark DataFrames")
print("  Output rendered within Databricks notebook")
print("  Source: apex_retail.GOLD_tables (Star Schema)")
print("")
print("="*60)
print("✓ Phase 5 + 6 delivered: Gold Star Schema + KPI Reporting")
print("="*60)


   PHASE 6: BUSINESS REPORTING (KPI GENERATION) COMPLETE

  KPI 1: Net Margin by Region              ✓
  KPI 2: AOV by Promotion                  ✓
  KPI 3: Demographic Churn Heatmap         ✓
  KPI 4: Product Quality Index             ✓
  KPI 5: Store Traffic by Day of Week      ✓

  All KPIs computed using PySpark DataFrames
  Output rendered within Databricks notebook
  Source: apex_retail.GOLD_tables (Star Schema)

✓ Phase 5 + 6 delivered: Gold Star Schema + KPI Reporting
